In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..')) 

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from src.features import FeatureEngineer
from src.regime import RegimeDetector
from src.strategies.trend_engine import TrendEngine

# --- SETUP DATA & REGIMES ---
df = pd.read_parquet('../data/raw/SPY.parquet')
df = df[~df.index.duplicated(keep='first')]

# Engineer
fe = FeatureEngineer(df)
fe.add_volatility_features().add_trend_features().add_volume_features()
features = fe.get_features()

# Detect Regimes
regime_engine = RegimeDetector(n_components=4)
# IMPORTANT: Map the columns correctly as per your previous output
regime_engine.feature_cols = ['Vol_ratio', 'Momentum', 'Vol_short']
regime_engine.fit(features)
probs = regime_engine.predict_proba(features)

# --- SIMULATION LOOP ---
# We use a safety buffer greater than the lookback window (50)
lookback_window = 50
buffer = 60 # Give it 10 extra bars to minimize NaN issues

engine = TrendEngine(lookback=lookback_window)
signals = []
equity = [10000]
position = 0
entry_price = 0

# Align indices
aligned_df = fe.df.loc[features.index]

# Start loop only when we have enough data (buffer)
for i in range(buffer, len(aligned_df)):
    
    # Slice: Get the last 'buffer' rows leading up to time 'i'
    # Start index must be non-negative
    start_idx = i - buffer
    market_slice = aligned_df.iloc[start_idx:i] 
    
    # Check probabilities at time 'i'
    current_probs = probs[i] # [P(0), P(1), P(2), P(3)]
    
    # Ask Engine for Signal
    try:
        signal = engine.generate_signal(market_slice, current_probs)
    except Exception as e:
        print(f"Error at index {i}: {e}")
        continue
    
    # Simple Execution Logic
    price = market_slice['Close'].iloc[-1]
    date = market_slice.index[-1]
    
    if signal['action'] == 'BUY' and position == 0:
        position = 1
        entry_price = price
        engine.position = 1 # Update internal state
        signals.append((date, 'BUY', price))
        
    elif signal['action'] == 'SELL' and position == 1:
        position = 0
        pnl = (price - entry_price) / entry_price
        equity.append(equity[-1] * (1 + pnl))
        engine.position = 0
        signals.append((date, 'SELL', price))

# Results
print(f"Final Equity: ${equity[-1]:.2f}")
print(f"Total Trades: {len(signals)//2}")
print(f"Return: {((equity[-1]-10000)/10000)*100:.2f}%")

Model trained. Converged: True


IndexError: single positional indexer is out-of-bounds